# Spellchecker quest

Хтось наробив помилок у віршах Тараса Шевченка. Ваша задача -- виправити ці помилки і прочитати приховане повідомлення.

## Задача

Ви отримаєте тренувальні та тестувальні дані.

Тренувальні дані знаходяться в полі `lab.train_text`. Це звичайний нерозмічений текст. На ньому необхідно натренувати мовну модель. Підійде будь-яка: n-грамна, feed-forward нейронну модель або їх ансамбль. Токенізацію раджу робити по літерах або BPE.

Тестувальні дані знаходиться в полі `lab.test_items`. Приклад одного елемента:

```json
{
  "text": "Співали б прозу, та по ножах,",
  "error_start": 23,
  "error_end": 28,
  "error": "ножах",
  "corrections": [
    "ногах",
    "йотах",
    "єнотах",
    "ножах",
    "нотах"
  ]
}
```

`error_start` та `error_end` вказують на місцезнаходження помилки в тексті (в символах). У цьому прикладі помилкою є `text[23:28]`, тобто слово "ножах".

`corrections` -- це список можливих виправлень.

Ваша задача -- натренувати мовну модель та за її допомогою обрати правильне виправлення серед запропонованих.


## Приховане повідомлення

Один приклад в `lab.test_items` дає можливість прочитати одну літеру прихованого повідомлення. Для цього знайдіть різницю між літерами слова з помилкою (`error`) та обраним моделлю виправленням. Надрукуйте цю літеру. Якщо слово з помилкою насправді правильне, а таке теж буває, надрукуйте пробіл. Приклади:

```
Error               Correction     Надрукувати
-------------------------------------------
привіт               приліт        л
пні                  поні          о
баллет               балет         л
привіт               привіт        [пробіл]
```

Приховане повідомлення, яке ви побачите в результаті -- це рядок з віршу одного з українських авторів.

Відповідь на квест -- ім'я автора/ки у форматі "Ім'я Прізвище".

Полетіли! 🚀

In [ ]:
!pip install --quiet --upgrade https://static.lp-nlp.com/pypi/lpnlp-2026.10.1-py3-none-any.whl

In [ ]:
import lpnlp

lab = lpnlp.start(
    email="user.name@lpnu.ua",                   # <----------- Заповніть це поле
    lab="quest_spellchecker"
    )

In [ ]:
# Якщо вам потрібні тренувальні дані у вигляді файлу на вашому комп'ютері,
# розкоментуйте цей код:

# with open("train.txt", "wt") as f:
#     f.write(lab.train_text)

# from google.colab import files
# files.download("train.txt")

## Мовна модель

Натренуйте свою мовну модель тут

In [ ]:
print(lab.train_text[:330])

In [ ]:
############################################

##### Ваш код для тренування моделі тут

vocab = ...
model = ...

...

############################################

# Читаємо між рядків

In [ ]:
import math
import collections
from typing import List, Tuple


# Допоміжна функція:
def get_letter(error: str, correction: str) -> str:
    """Повертає літеру повідомлення або пробіл для незміненого слова."""
    error = error.lower()
    correction = correction.lower()

    if error == correction:
        return " "

    added = collections.Counter(correction) - collections.Counter(error)
    removed = collections.Counter(error) - collections.Counter(correction)
    difference = added or removed

    if sum(difference.values()) != 1:
        raise ValueError(
            f"Очікувалася різниця в одну літеру: {error!r} → {correction!r}"
        )

    return next(iter(difference))

assert get_letter("привіт", "приліт") == "л"
assert get_letter("пні", "поні") == "о"
assert get_letter("баллет", "балет") == "л"
assert get_letter("Привіт", "привіт") == " "


def score_text(text: str, model, vocab) -> float:
    """
    Повертає perplexity або negative log-likelihood для тексту
    (менше -- краще).
    """

    ####################################################
    return 0.42     # Заглушка. Напишіть справжню фунцію
    ####################################################


# Можете змінювати параметри та весь цей код, якщо потрібно
def solve(model, vocab, test_items) -> Tuple[List[str], str]:
    """Повертає список виправлених слів для кожного з текстів в test_items та
    секретне повідомлення.
    """

    choices = []
    secret = []

    for item in test_items:
        scores = []
        for corr in item['corrections']:

            # Підставляємо слово-кандидат в текст
            text = item['text'][:item['error_start']] + corr + item['text'][item['error_end']:]

            # Рахуємо score тексту
            score = score_text(text, model, vocab)
            scores.append(score)

            # print(f'{score:.4f} {text}')

        # Сортуємо кандидатів на виправлення за score
        result = sorted(zip(scores, item['corrections']), key=lambda x: x[0])

        # Обираємо найкращу заміну
        best = result[0]
        best_word = best[1]
        choices.append(best_word)

        # Знаходимо чергову літеру повідомлення
        error = item['error']
        letter = get_letter(error, best_word)
        secret.append(letter)

    secret_message = ''.join(secret)

    return choices, secret_message

choices, secret_message = solve(model, vocab, lab.test_items)

lab.evaluate_accuracy(choices)
print("SECRET MESSAGE: ", secret_message)


In [ ]:
lab.answer("Ім'я Прізвище")   # Замініть на ім'я автора/ки розшифрованих рядків

# Perplexity

Окрім точності виправлення помилок, оцінимо, наскільки добре ваша модель передбачає текст, якого вона не бачила під час навчання.
Для цього в пакеті є файл `valid.txt`. Його вміст доступний через `lab.valid_text`. Цей текст не входить до тренувальних даних.

Не тренуйте модель або токенізатор на `lab.valid_text`! Використовуйте його лише для оцінювання.

### Що порахувати

Токенізуйте `lab.valid_text` тим самим токенізатором, що й тренувальні дані. Для кожного токена знайдіть імовірність, яку модель надає йому з урахуванням попереднього контексту.

Порахуйте:

1. `N_tokens` — кількість текстових токенів, для яких обчислено ймовірність.
2. `total_NLL` — сумарний від’ємний логарифм їхніх імовірностей:

   ```text
   total_NLL = -Σ ln p(token_i | попередній контекст)
   ```

3. Perplexity на токен:

   ```python
   perplexity = math.exp(total_NLL / N_tokens)
   ```

Менша perplexity означає, що модель краще передбачає цей текст.

Perplexity на токен залежить від токенізації. Наприклад, не можна напряму порівнювати perplexity моделі на літерах із perplexity моделі на BPE-токенах.

Тому додатково порахуйте

4. Perplexity на символ:

```python
N_chars = len(lab.valid_text)
character_perplexity = math.exp(total_NLL / N_chars)
```

У `N_chars` враховуйте весь початковий текст, включно з пробілами та символами нового рядка. Для обох метрик використовуйте той самий `total_NLL`.

Для коректного порівняння моделі мають оцінювати той самий текст. Якщо ви замінюєте невідомі слова на `<unk>`, переводите текст у нижній регістр або викидаєте розділові знаки, ви оцінюєте змінений текст — обов’язково зазначте це у звіті.


In [ ]:
# @title Звіт

# Тип моделі (наприклад, символьна 5-грамна або feed-forward нейронна)
# та розмір контексту в токенах.
model_and_context = "" # @param {type:"string"}

# Тип токенізації (літери / BPE / слова) та кількість токенів у словнику.
tokenization_and_vocab = "" # @param {type:"string"}

# Кількість текстових токенів із valid.txt, для яких обчислено ймовірність.
# Не враховуйте padding, додані службові токени та повторно оцінений контекст.
n_tokens = 0 # @param {type:"integer"}

# Кількість символів у початковому тексті: len(lab.valid_text).
# Враховуйте пробіли та символи нового рядка.
n_chars = 0 # @param {type:"integer"}

# Сумарний від’ємний натуральний логарифм імовірностей токенів:
# total_NLL = -Σ ln p(token_i | контекст)
total_nll = 0.0 # @param {type:"number"}

# Perplexity на токен: exp(total_nll / n_tokens).
token_perplexity = 0.0 # @param {type:"number"}

# Perplexity на символ: exp(total_nll / n_chars).
# Використовуйте той самий total_nll, що й для perplexity на токен.
character_perplexity = 0.0 # @param {type:"number"}

# Опишіть зміну регістру, обробку невідомих токенів, пробілів і пунктуації,
# використання BOS/EOS, поділ на вікна та обробку їхніх меж.
processing_notes = "" # @param {type:"string"}


📝  Відправте посилання на цей colab або PDF з ним у відповідне завдання VNS або на email викладачу, що приймає лабораторні
